# Even k, Two Classes, and a Coin Flip

**Topic:** k-Nearest Neighbours &nbsp;·&nbsp; **Difficulty:** Medium

With `k = 8` and two classes, a 4-4 split happens often, and whatever your
implementation does about it (pick the lowest class index, pick the first one
`argmax` finds) is an arbitrary decision made by a library author, not by you.

Distance weighting removes the problem entirely: a neighbour two units away
should not count as much as one right next door, and once votes are weighted by
`1 / (distance + 1e-12)` exact ties essentially stop happening.

## The data (created in the next cell)

- `X_train` `(900, 6)`, `y_train` `(900,)`: two classes.
- `X_test` `(500, 6)`, `y_test` for grading.
- `k = 8`, deliberately even.

## Your task

With **NumPy only** (no `sklearn.neighbors`):

1. Standardize using training statistics.
2. For each test point find the `k = 8` nearest training points by Euclidean
   distance.
3. `pred_uniform`: plain majority vote, breaking ties toward class **0**.
4. `pred_weighted`: vote weighted by `1 / (distance + 1e-12)`.
5. Set `answers` to a dict with `"n_ties"` (int, test points with a 4-4 split),
   `"acc_uniform"`, `"acc_weighted"`: accuracies against `y_test`.

## Evaluation

Compared to the reference within `1e-6 * (1 + |ref|)`, and the evaluation
cell also requires that ties actually occur (`n_ties` > 0).

In [ ]:
import numpy as np

rng = np.random.default_rng(707)
c0 = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
c1 = np.array([1.0, 0.9, -0.8, 0.6, 0.0, 0.0])

def make(n, seed):
    r = np.random.default_rng(seed)
    y = r.integers(0, 2, n)
    X = np.where(y[:, None] == 0, c0, c1) + r.normal(0, 1.05, (n, 6))
    X[:, 5] *= 60.0                      # one loud, useless feature
    return X, y

X_train, y_train = make(900, 31)
X_test, y_test = make(500, 32)
k = 8
print(f"train {X_train.shape}, test {X_test.shape}, k = {k} (even)")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("n_ties", [], 67.0, -11.218346992571444, 1e-06),
    ("acc_uniform", [], 0.724, -0.12122512272569738, 1e-06),
    ("acc_weighted", [], 0.722, -0.12089024669606839, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "n_ties" in answers:
    _t = answers["n_ties"] > 0
    print(f"  {'ties actually occur':<24} {'ok' if _t else 'wrong'}  ({int(answers['n_ties'])} of 500)")
    _ok = _ok and _t

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

Ties are common here. With `k = 8` and heavily overlapping classes, about one test point in
eight (67 of 500) splits 4-4, and every one of those is decided by an
implementation detail. Weighting by inverse distance makes the vote continuous,
so the decision depends on the geometry rather than on a tiebreak rule. Note the
standardization also matters: feature 5 is 60x louder and carries no signal.

```python
import numpy as np

mu, sd = X_train.mean(axis=0), X_train.std(axis=0)
Ztr, Zte = (X_train - mu) / sd, (X_test - mu) / sd

D = np.linalg.norm(Zte[:, None, :] - Ztr[None, :, :], axis=2)
nn = np.argsort(D, axis=1)[:, :k]
lab = y_train[nn]
dist = np.take_along_axis(D, nn, axis=1)

votes1 = (lab == 1).sum(axis=1)
n_ties = int((votes1 == k // 2).sum())
pred_uniform = (votes1 > k // 2).astype(int)      # ties -> class 0

w = 1.0 / (dist + 1e-12)
w1 = (w * (lab == 1)).sum(axis=1)
w0 = (w * (lab == 0)).sum(axis=1)
pred_weighted = (w1 > w0).astype(int)

answers = {
    "n_ties": n_ties,
    "acc_uniform": float((pred_uniform == y_test).mean()),
    "acc_weighted": float((pred_weighted == y_test).mean()),
}
```